[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/optimization/08_stochastic_optimization_for_ml/exercises.ipynb)

# Topic 08: Stochastic Optimization for Machine Learning — Exercises

## Level 0 — Concept Check

### Problem L0.1: Is the Mini-Batch Gradient Unbiased?

**Problem Statement:** Prove that the mini-batch gradient
$\mathbf{g}_{\mathcal{B}} = \frac{1}{B}\sum_{i\in\mathcal{B}}\nabla f_i(\mathbf{x})$ is an unbiased
estimator of $\nabla f = \frac{1}{N}\sum_i\nabla f_i$ when $\mathcal{B}$ is drawn uniformly at random.
Does the conclusion change if the batch is the *same* fixed subset every step?

*Intuition:* The finite sum is already an expectation under the uniform distribution over examples, so sampling from that distribution reproduces it exactly.

**Solution:**

**Step 1 (one uniform draw).** Let $i$ be uniform on $\{1,\dots,N\}$. Then

$$
\mathbb{E}\left[\nabla f_i(\mathbf{x})\right] = \sum_{j=1}^{N}\frac{1}{N}\,\nabla f_j(\mathbf{x}) = \nabla f(\mathbf{x})
$$

**Step 2 (a batch of $B$).** By linearity of expectation, whether or not the draws are independent,

$$
\mathbb{E}\left[\mathbf{g}_{\mathcal{B}}\right] = \frac{1}{B}\sum_{r=1}^{B}\mathbb{E}\left[\nabla f_{i_r}(\mathbf{x})\right] = \nabla f(\mathbf{x})
$$

Unbiasedness needs only that each *marginal* is uniform — sampling with or without replacement both
qualify. Independence matters for the *variance*, not the mean.

**Step 3 (a fixed subset is biased).** If $\mathcal{B} = \mathcal{B}_0$ is fixed once and for all, then
$\mathbf{g}_{\mathcal{B}_0}$ is deterministic and equals

$$
\frac{1}{B}\sum_{i\in\mathcal{B}_0}\nabla f_i(\mathbf{x}) = \nabla f_{\mathcal{B}_0}(\mathbf{x}) \neq \nabla f(\mathbf{x}) \quad \text{in general}
$$

This is not SGD on $f$ at all: it is exact gradient descent on the *sub-problem* $f_{\mathcal{B}_0}$, and it
will converge to the minimizer of that sub-problem — the mathematical description of overfitting a fixed
mini-batch.

**Step 4 (shuffling in practice).** Real training uses **random reshuffling**: permute the data each epoch
and sweep. Within an epoch the batches are not independent and each individual batch gradient is *not*
unbiased conditionally, but the scheme is unbiased over a full epoch and provably converges faster than
with-replacement sampling for large $N$.

$$
\boxed{\mathbb{E}\left[\mathbf{g}_{\mathcal{B}}\right] = \nabla f(\mathbf{x}) \ \text{for any uniformly sampled batch}; \ \text{a fixed batch is biased}}
$$

> **Key takeaway:** Unbiasedness is a statement about *resampling*, not about batch size — a batch of one is as unbiased as a batch of a thousand, and only the variance distinguishes them.

### Problem L0.2: Does Constant-Step SGD Reach the Minimizer?

**Problem Statement:** True or false: SGD with a small enough constant step size $\alpha$ converges to
$\mathbf{x}^*$ on a strongly convex objective. Justify with the scalar model
$f(x) = \frac{\mu}{2}x^2$ and gradient noise of variance $\sigma^2$.

*Intuition:* At the optimum the true gradient vanishes but the estimate does not, so the iterate keeps being kicked away.

**Solution:**

**False.** The iterates converge *in distribution* to a stationary law concentrated in a ball around
$\mathbf{x}^*$, never to the point itself.

**Step 1 (the scalar recursion).** With $g_k = \mu x_k + \epsilon_k$, $\mathbb{E}[\epsilon_k]=0$,
$\mathbb{E}[\epsilon_k^2]=\sigma^2$:

$$
x_{k+1} = x_k - \alpha\left(\mu x_k + \epsilon_k\right) = (1-\alpha\mu)x_k - \alpha\epsilon_k
$$

an AR(1) process with $\rho = 1-\alpha\mu$.

**Step 2 (the mean does converge).** $\mathbb{E}[x_{k+1}] = \rho\,\mathbb{E}[x_k]$, so
$\mathbb{E}[x_k] = \rho^k x_0 \to 0$ for $0 \lt \alpha \lt 2/\mu$. The *bias* dies geometrically.

**Step 3 (the variance does not).** Squaring and using independence of $\epsilon_k$ from $x_k$:

$$
\mathbb{E}\left[x_{k+1}^2\right] = \rho^2\,\mathbb{E}\left[x_k^2\right] + \alpha^2\sigma^2 \ \xrightarrow[k\to\infty]{}\ v_\infty = \frac{\alpha^2\sigma^2}{1-\rho^2} = \frac{\alpha^2\sigma^2}{2\alpha\mu - \alpha^2\mu^2} \approx \frac{\alpha\sigma^2}{2\mu}
$$

for small $\alpha$. So $\mathbb{E}[x_k^2] \to \frac{\alpha\sigma^2}{2\mu} \gt 0$: the iterate rattles
forever inside a ball of radius $\sqrt{\alpha\sigma^2/(2\mu)}$.

**Step 4 (matching the general theorem).** In function value,
$\mathbb{E}[f(x_k)] - f^* = \frac{\mu}{2}v_\infty \approx \frac{\alpha\sigma^2}{4}$, in agreement (up to the
constant) with the general bound $\frac{L\alpha\sigma^2}{2\mu B}$ of the theory notebook.

**Step 5 (the three fixes).** Shrink $\alpha$ (staircase decay), enlarge $B$ (halves $\sigma^2/B$), or
reduce $\sigma$ itself with a control variate (SVRG). All three appear as the factor
$\alpha\sigma^2/(\mu B)$ in the floor.

$$
\boxed{\mathbb{E}\left[x_k^2\right]\to\frac{\alpha^2\sigma^2}{1-(1-\alpha\mu)^2}\approx\frac{\alpha\sigma^2}{2\mu} \gt 0: \ \text{a noise ball, not a point}}
$$

> **Key takeaway:** Constant-step SGD is a *sampler*, not a minimizer: it converges to a stationary distribution whose width is set by $\alpha\sigma^2/(\mu B)$, and only a schedule collapses that distribution to a point.

### Problem L0.3: Which Schedules Are Admissible?

**Problem Statement:** For each schedule, decide whether it satisfies the Robbins-Monro conditions
$\sum_k\alpha_k = \infty$ and $\sum_k\alpha_k^2 \lt \infty$:
(a) $\alpha_k = \alpha_0$; (b) $\alpha_k = \alpha_0/k$; (c) $\alpha_k = \alpha_0/\sqrt{k}$;
(d) $\alpha_k = \alpha_0/k^2$; (e) $\alpha_k = \alpha_0\,(0.99)^k$. State the general rule for
$\alpha_k = \alpha_0 k^{-p}$.

*Intuition:* One condition demands enough total travel, the other demands finite total injected noise.

**Solution:**

Recall $\sum_k k^{-q}$ diverges iff $q \le 1$.

**(a) Constant $\alpha_0$.** $\sum\alpha_k = \infty$ $\checkmark$ but $\sum\alpha_k^2 = \infty$ $\times$.
**Not admissible** — this is exactly the noise ball of Problem L0.2.

**(b) $\alpha_0/k$.** $\sum 1/k = \infty$ $\checkmark$ and $\sum 1/k^2 = \pi^2/6 \lt \infty$ $\checkmark$.
**Admissible**, and the canonical choice for strongly convex problems.

**(c) $\alpha_0/\sqrt{k}$.** $\sum k^{-1/2} = \infty$ $\checkmark$ but $\sum k^{-1} = \infty$ $\times$.
**Not admissible** in the strict sense. It is nevertheless the standard choice for *general convex*
problems, where the analysis uses the averaged iterate and only needs the ratio
$\frac{\sum\alpha_k^2}{\sum\alpha_k}\to0$, which $k^{-1/2}$ satisfies (giving the $O(\log K/\sqrt{K})$ rate
of Problem L3.2).

**(d) $\alpha_0/k^2$.** $\sum k^{-2} \lt \infty$ $\times$ (first condition fails) and
$\sum k^{-4} \lt \infty$ $\checkmark$. **Not admissible**: total travel is bounded by
$\alpha_0\pi^2/6$, so the iterate can stall permanently at distance greater than that from
$\mathbf{x}^*$.

**(e) Geometric $\alpha_0(0.99)^k$.** $\sum\alpha_k = 100\alpha_0 \lt \infty$ $\times$. **Not admissible**,
and the failure is severe: the iterate freezes wherever it happens to be.

**General rule for $\alpha_k = \alpha_0k^{-p}$.**

$$
\sum_k\alpha_k = \infty \iff p \le 1, \qquad \sum_k\alpha_k^2 \lt \infty \iff 2p \gt 1 \iff p \gt \tfrac12
$$

$$
\boxed{\alpha_k = \alpha_0 k^{-p} \ \text{is Robbins-Monro admissible} \iff p \in \left(\tfrac12,\ 1\right]}
$$

> **Key takeaway:** Every practical schedule (step decay, cosine, inverse-square-root) is an engineering approximation to this window; a schedule that decays too fast is a more common failure in practice than one that decays too slowly.

### Problem L0.4: Why Adam Needs Bias Correction

**Problem Statement:** Adam initializes $\mathbf{m}_0 = \mathbf{v}_0 = \mathbf{0}$. Show that under
stationary gradients this makes $\mathbf{v}_k$ biased low by the factor $1-\beta_2^k$, quantify the bias
at $k = 1, 10, 100, 1000$ for $\beta_2 = 0.999$, and explain what would go wrong without the correction.

*Intuition:* The moving average starts from an artificial zero and needs many steps to "fill up"; dividing by how full it is restores the right scale.

**Solution:**

**Step 1 (unroll).** From $\mathbf{v}_k = \beta_2\mathbf{v}_{k-1} + (1-\beta_2)\mathbf{g}_k^{\odot2}$ with
$\mathbf{v}_0 = \mathbf{0}$,

$$
\mathbf{v}_k = (1-\beta_2)\sum_{t=1}^{k}\beta_2^{\,k-t}\,\mathbf{g}_t^{\odot2}
$$

**Step 2 (expectation under stationarity).** If $\mathbb{E}[\mathbf{g}_t^{\odot2}] = \mathbb{E}[\mathbf{g}^{\odot2}]$
for all $t$, the geometric sum gives

$$
\mathbb{E}\left[\mathbf{v}_k\right] = (1-\beta_2)\,\frac{1-\beta_2^{\,k}}{1-\beta_2}\;\mathbb{E}\left[\mathbf{g}^{\odot2}\right] = \left(1-\beta_2^{\,k}\right)\mathbb{E}\left[\mathbf{g}^{\odot2}\right]
$$

so dividing by $1-\beta_2^k$ makes $\hat{\mathbf{v}}_k$ unbiased.

**Step 3 (numbers for $\beta_2 = 0.999$).**

| $k$ | $1-\beta_2^{\,k}$ | correction factor $1/(1-\beta_2^k)$ |
|---|---|---|
| $1$ | $0.001$ | $1000$ |
| $10$ | $0.00995$ | $100.5$ |
| $100$ | $0.0952$ | $10.5$ |
| $1000$ | $0.632$ | $1.58$ |
| $10000$ | $0.99995$ | $1.00005$ |

The bias is enormous early and is essentially gone after a few thousand steps — precisely the horizon
$1/(1-\beta_2) = 1000$.

**Step 4 (what goes wrong without it).** The update is $\alpha\hat{m}/(\sqrt{\hat{v}}+\varepsilon)$. Using
uncorrected moments with a constant gradient $g$:

$$
\frac{m_1}{\sqrt{v_1}} = \frac{(1-\beta_1)g}{\sqrt{(1-\beta_2)g^2}} = \frac{0.1}{\sqrt{0.001}} \approx 3.16
$$

so the first step is $3.16\alpha$ rather than $\alpha$ — and the mismatch is worse for other
$(\beta_1,\beta_2)$ pairs. Because $\beta_1 \ll \beta_2$ in the usual defaults, $m$ fills faster than $v$,
so the ratio *overshoots*; the correction restores the intended unit-scale step from iteration one.

$$
\boxed{\mathbb{E}\left[\mathbf{v}_k\right] = \left(1-\beta_2^{\,k}\right)\mathbb{E}\left[\mathbf{g}^{\odot2}\right] \implies \hat{\mathbf{v}}_k = \frac{\mathbf{v}_k}{1-\beta_2^{\,k}}}
$$

> **Key takeaway:** The correction is a pure initialization artefact — it has nothing to do with gradient statistics and everything to do with starting an exponential average at zero.

## Level 1 — Foundation

### Problem L1.1: Computing the Gradient Noise of a Finite Sum

**Problem Statement:** At a point $x$, a four-example problem has per-example derivatives
$f_1' = 3$, $f_2' = 1$, $f_3' = -1$, $f_4' = -3$. Compute $f'(x)$, the per-example variance $\sigma^2$, and
the mini-batch variance for $B = 1, 2, 4$ both with and without replacement.

*Intuition:* Variance is the average squared deviation of individual gradients from their mean, and batching averages it down.

**Solution:**

**Step 1 (the full gradient).**

$$
f'(x) = \frac{3+1-1-3}{4} = 0
$$

so $x$ is a stationary point of the *empirical* objective, yet no individual example agrees.

**Step 2 (per-example variance).**

$$
\sigma^2 = \frac{1}{4}\sum_{i=1}^{4}\left(f_i' - f'\right)^2 = \frac{9+1+1+9}{4} = 5
$$

**Step 3 (with replacement, $\sigma^2/B$).**

| $B$ | variance $\sigma^2/B$ | std. dev. |
|---|---|---|
| $1$ | $5$ | $2.236$ |
| $2$ | $2.5$ | $1.581$ |
| $4$ | $1.25$ | $1.118$ |

Note that even at $B = N = 4$ the with-replacement estimator is **not** exact — repeated indices leave
residual variance $1.25$.

**Step 4 (without replacement, finite-population correction).**
$\frac{\sigma^2}{B}\cdot\frac{N-B}{N-1}$ with $N = 4$:

| $B$ | correction $\frac{4-B}{3}$ | variance |
|---|---|---|
| $1$ | $1$ | $5$ |
| $2$ | $2/3$ | $1.667$ |
| $4$ | $0$ | $0$ |

At $B = N$ the estimator becomes the exact full gradient, as it must.

**Step 5 (the practical reading).** At $B=1$ the standard deviation $2.236$ dwarfs the signal $f' = 0$: the
step is pure noise. At $B=2$ (without replacement) the variance has already fallen by a factor $3$. In
general the signal-to-noise ratio of a step is

$$
\mathrm{SNR} = \frac{\lVert \nabla f\rVert^2}{\sigma^2/B} = \frac{B\lVert \nabla f\rVert^2}{\sigma^2}
$$

which is exactly the quantity the gradient-noise-scale heuristic monitors to choose $B$.

$$
\boxed{\sigma^2 = 5; \ \text{with replacement } \tfrac{5}{B}; \ \text{without replacement } \tfrac{5}{B}\cdot\tfrac{4-B}{3}}
$$

> **Key takeaway:** Sampling without replacement is strictly better at equal $B$, and the gap is largest when $B$ is a sizeable fraction of $N$ — which is why epoch-based reshuffling is the default in every training loop.

### Problem L1.2: Sizing the Noise Ball

**Problem Statement:** A training problem has $L = 10$, $\mu = 1$, per-example gradient variance
$\sigma^2 = 4$, and batch size $B = 16$. Using
$\delta_\infty = \frac{L\alpha\sigma^2}{2\mu B}$, compute the noise floor at $\alpha = 0.05$, find the
largest $\alpha$ achieving $\delta_\infty \le 10^{-3}$, and estimate the iterations per decimal digit at
that $\alpha$.

*Intuition:* The floor is linear in $\alpha$ and the transient rate is also linear in $\alpha$ — accuracy and speed are directly traded.

**Solution:**

**Step 1 (batch variance).**

$$
\sigma_B^2 = \frac{\sigma^2}{B} = \frac{4}{16} = 0.25
$$

**Step 2 (floor at $\alpha = 0.05$).**

$$
\delta_\infty = \frac{L\alpha\sigma_B^2}{2\mu} = \frac{10\cdot 0.05\cdot 0.25}{2\cdot 1} = 0.0625
$$

Also check stability: $\alpha = 0.05 \le 1/L = 0.1$ $\checkmark$.

**Step 3 (largest $\alpha$ for a floor of $10^{-3}$).**

$$
\alpha \le \frac{2\mu\,\delta_{\text{target}}}{L\sigma_B^2} = \frac{2\cdot1\cdot10^{-3}}{10\cdot0.25} = 8\times10^{-4}
$$

**Step 4 (transient speed at $\alpha = 8\times10^{-4}$).** The contraction factor is
$1-\alpha\mu = 0.9992$, so one decimal digit of suboptimality reduction costs

$$
k = \frac{\ln 10}{-\ln(0.9992)} \approx \frac{2.3026}{8.0032\times10^{-4}} \approx 2877 \ \text{iterations}
$$

versus $\approx 46$ iterations per digit at $\alpha = 0.05$ — a $63\times$ slowdown bought for a $63\times$
lower floor. That is the trade in its rawest form.

**Step 5 (a better plan: staircase).** Run at $\alpha = 0.05$ until the loss plateaus near $0.0625$
(about $2$ digits, $\approx 100$ steps), then cut $\alpha$ by $10$ repeatedly. Each cut lowers the floor by
$10$ and costs only $\approx 10\times$ more steps *for that phase*, and the total is dominated by the last
phase — far cheaper than starting at $8\times10^{-4}$. Equivalently: increase $B$. Going to $B = 1024$
multiplies the affordable $\alpha$ by $64$ at the same floor.

$$
\boxed{\delta_\infty(\alpha{=}0.05) = 0.0625; \quad \alpha \le 8\times10^{-4} \ \text{for} \ \delta_\infty \le 10^{-3}; \quad \approx 2877 \ \text{iterations/digit}}
$$

> **Key takeaway:** Learning-rate decay is not a heuristic — it is the only way (short of raising $B$ or reducing $\sigma$) to move the floor, and the staircase shape follows from the linearity of $\delta_\infty$ in $\alpha$.

### Problem L1.3: Verifying a Schedule and Its Partial Sums

**Problem Statement:** For $\alpha_k = \frac{\alpha_0}{k}$, compute the leading asymptotics of
$\sum_{k=1}^{K}\alpha_k$ and $\sum_{k=1}^{K}\alpha_k^2$, and use them to explain why this schedule sits at
the exact boundary of admissibility. Then do the same for $\alpha_k = \alpha_0/\sqrt{k}$ and interpret the
ratio $\frac{\sum\alpha_k^2}{\sum\alpha_k}$.

*Intuition:* The first sum measures total distance the drift can cover; the second measures total injected noise energy; their ratio is the residual error the analysis predicts.

**Solution:**

**Step 1 (harmonic schedule, $p = 1$).**

$$
\sum_{k=1}^{K}\frac{\alpha_0}{k} = \alpha_0\left(\ln K + \gamma + O(1/K)\right) \to \infty, \qquad \sum_{k=1}^{K}\frac{\alpha_0^2}{k^2} \to \alpha_0^2\frac{\pi^2}{6} \lt \infty
$$

Both Robbins-Monro conditions hold, but only just: the first diverges as slowly as possible (logarithmically)
while the second converges. Reducing $p$ below $1$ speeds the divergence; raising it above $1$ destroys it.

**Step 2 (the boundary).** For $\alpha_k = \alpha_0k^{-p}$:

$$
\sum_{k\le K}\alpha_k \sim \begin{cases}\alpha_0\frac{K^{1-p}}{1-p}, & p \lt 1\\ \alpha_0\ln K, & p = 1\end{cases}, \qquad \sum_{k\le K}\alpha_k^2 \sim \begin{cases}\alpha_0^2\frac{K^{1-2p}}{1-2p}, & p \lt \tfrac12\\ \alpha_0^2\ln K, & p = \tfrac12 \\ \text{const}, & p \gt \tfrac12\end{cases}
$$

so admissibility is exactly $\tfrac12 \lt p \le 1$, and $p=1$ is the fast end of that window.

**Step 3 (square-root schedule, $p = 1/2$).**

$$
\sum_{k=1}^K\frac{\alpha_0}{\sqrt{k}} \approx 2\alpha_0\sqrt{K}, \qquad \sum_{k=1}^K\frac{\alpha_0^2}{k} \approx \alpha_0^2\ln K
$$

The second sum diverges, so strict Robbins-Monro fails — yet the schedule is standard for general convex
problems.

**Step 4 (why the ratio is the right diagnostic).** The convex SGD bound for the averaged iterate
(Problem L3.2) is

$$
\mathbb{E}\left[f(\bar{\mathbf{x}}_K)\right]-f^* \le \frac{\lVert \mathbf{x}_0-\mathbf{x}^*\rVert^2 + G^2\sum_k\alpha_k^2}{2\sum_k\alpha_k}
$$

For $p = 1/2$ this gives

$$
\frac{D^2 + G^2\alpha_0^2\ln K}{4\alpha_0\sqrt{K}} = O\!\left(\frac{\log K}{\sqrt{K}}\right) \to 0
$$

so convergence holds even though $\sum\alpha_k^2$ diverges: what the analysis actually needs is
$\frac{\sum\alpha_k^2}{\sum\alpha_k} \to 0$, a weaker requirement than $\sum\alpha_k^2 \lt \infty$.

$$
\boxed{p=1: \ \sum\alpha_k \sim \alpha_0\ln K, \ \sum\alpha_k^2 \to \tfrac{\pi^2}{6}\alpha_0^2; \quad p=\tfrac12: \ \sum\alpha_k \sim 2\alpha_0\sqrt{K}, \ \frac{\sum\alpha_k^2}{\sum\alpha_k} = O\!\left(\tfrac{\log K}{\sqrt K}\right)}
$$

> **Key takeaway:** Robbins-Monro is sufficient, not necessary; the operative quantity in every modern proof is the ratio of the two partial sums, which is what makes $1/\sqrt{k}$ and cosine schedules legitimate.

### Problem L1.4: Three AdaGrad Steps by Hand

**Problem Statement:** Run AdaGrad with $\alpha = 0.1$, $\varepsilon = 10^{-8}$ on a two-coordinate problem
whose observed gradients are $\mathbf{g}_1 = (3,\ 0.1)$, $\mathbf{g}_2 = (1,\ 0.1)$,
$\mathbf{g}_3 = (0.5,\ 0.1)$. Report $\mathbf{v}_k$ and the per-coordinate displacement at each step, and
explain the behaviour of the small-gradient coordinate.

*Intuition:* Dividing by the accumulated root-mean-square makes the first step always $\pm\alpha$, and thereafter rewards coordinates whose gradients have been small.

**Solution:**

AdaGrad: $v_{k,j} = \sum_{t\le k}g_{t,j}^2$, displacement
$\Delta_{k,j} = -\alpha\,g_{k,j}/\left(\sqrt{v_{k,j}}+\varepsilon\right)$.

**Step 1 ($k=1$).** $\mathbf{v}_1 = (9,\ 0.01)$, $\sqrt{\mathbf{v}_1} = (3,\ 0.1)$:

$$
\Delta_{1} = -0.1\left(\frac{3}{3},\ \frac{0.1}{0.1}\right) = (-0.1,\ -0.1)
$$

Both coordinates move by exactly $\alpha$, despite gradients differing by a factor $30$. The **first
AdaGrad step is always $\pm\alpha$ per coordinate** — the algorithm is scale-free.

**Step 2 ($k=2$).** $\mathbf{v}_2 = (9+1,\ 0.01+0.01) = (10,\ 0.02)$,
$\sqrt{\mathbf{v}_2} = (3.1623,\ 0.14142)$:

$$
\Delta_2 = -0.1\left(\frac{1}{3.1623},\ \frac{0.1}{0.14142}\right) = (-0.03162,\ -0.07071)
$$

**Step 3 ($k=3$).** $\mathbf{v}_3 = (10.25,\ 0.03)$, $\sqrt{\mathbf{v}_3} = (3.2016,\ 0.17321)$:

$$
\Delta_3 = -0.1\left(\frac{0.5}{3.2016},\ \frac{0.1}{0.17321}\right) = (-0.01562,\ -0.05774)
$$

**Summary.**

| $k$ | $v_{k,1}$ | $v_{k,2}$ | $\Delta_{k,1}$ | $\Delta_{k,2}$ |
|---|---|---|---|---|
| $1$ | $9$ | $0.01$ | $-0.1$ | $-0.1$ |
| $2$ | $10$ | $0.02$ | $-0.03162$ | $-0.07071$ |
| $3$ | $10.25$ | $0.03$ | $-0.01562$ | $-0.05774$ |

**Step 4 (interpretation).** Coordinate 2, whose gradients are tiny but *persistent*, retains a much larger
effective step ($-0.0577$ versus $-0.0156$) because its accumulator grows slowly. This is exactly the
behaviour that makes AdaGrad excel with sparse features: a rare word's embedding accumulates almost no
squared gradient and therefore keeps a large learning rate whenever it does appear.

**Step 5 (the built-in decay).** With gradients of roughly constant magnitude $\bar{g}_j$,
$v_{k,j}\approx k\bar{g}_j^2$, so the effective step is
$\alpha/(\bar{g}_j\sqrt{k}) = \Theta(k^{-1/2})$ — a Robbins-Monro-admissible schedule appearing for free.
The same property makes AdaGrad stall on very long runs, which is why RMSProp and Adam replace the sum by
an exponential moving average.

$$
\boxed{\Delta_1 = (-0.1,-0.1),\ \Delta_2 = (-0.0316,-0.0707),\ \Delta_3 = (-0.0156,-0.0577)}
$$

> **Key takeaway:** AdaGrad is a per-coordinate preconditioner *and* an automatic $1/\sqrt{k}$ schedule; separating those two roles is exactly what RMSProp does, and forgetting the second is why Adam needs an explicit decay.

### Problem L1.5: One Adam Step, With and Without Bias Correction

**Problem Statement:** With $\alpha = 10^{-3}$, $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\varepsilon = 10^{-8}$,
$\mathbf{m}_0 = \mathbf{v}_0 = 0$, and a constant scalar gradient $g = 0.1$, compute the first Adam update
with and without bias correction. Then compute the update at $k=2$ and comment.

*Intuition:* With a constant gradient the corrected update is exactly $\alpha$, whatever the gradient's magnitude — Adam normalizes the step, not the direction.

**Solution:**

**Step 1 ($k=1$, raw moments).**

$$
m_1 = (1-\beta_1)g = 0.1\times0.1 = 0.01, \qquad v_1 = (1-\beta_2)g^2 = 0.001\times0.01 = 10^{-5}
$$

**Step 2 (bias-corrected moments).**

$$
\hat{m}_1 = \frac{m_1}{1-\beta_1} = \frac{0.01}{0.1} = 0.1 = g, \qquad \hat{v}_1 = \frac{v_1}{1-\beta_2} = \frac{10^{-5}}{10^{-3}} = 10^{-2} = g^2
$$

Both estimates recover the true moments exactly, as they must after a single observation.

**Step 3 (the corrected update).**

$$
\Delta_1 = -\alpha\frac{\hat{m}_1}{\sqrt{\hat{v}_1}+\varepsilon} = -10^{-3}\cdot\frac{0.1}{0.1 + 10^{-8}} \approx -10^{-3} = -\alpha
$$

**A step of exactly $\alpha$** — this is why $\alpha$ in Adam is interpretable as a "trust region radius"
per coordinate rather than as a scale on the gradient.

**Step 4 (the uncorrected update).**

$$
\Delta_1^{\text{raw}} = -\alpha\frac{m_1}{\sqrt{v_1}+\varepsilon} = -10^{-3}\cdot\frac{0.01}{3.1623\times10^{-3}} = -3.162\times10^{-3} = -3.16\,\alpha
$$

$3.16$ times too large: the mean estimate has filled to $10\%$ of its target while the second-moment
estimate has filled to only $0.1\%$, and the ratio of the fill factors is
$\frac{1-\beta_1}{\sqrt{1-\beta_2}} = \frac{0.1}{0.0316} = 3.16$.

**Step 5 ($k=2$).** $m_2 = 0.9(0.01)+0.1(0.1) = 0.019$ and
$v_2 = 0.999\times10^{-5} + 0.001\times0.01 = 1.999\times10^{-5}$. Corrections:
$1-\beta_1^2 = 0.19$, $1-\beta_2^2 = 1.999\times10^{-3}$, so

$$
\hat{m}_2 = \frac{0.019}{0.19} = 0.1, \qquad \hat{v}_2 = \frac{1.999\times10^{-5}}{1.999\times10^{-3}} = 10^{-2}, \qquad \Delta_2 \approx -\alpha
$$

again exactly $\alpha$ — with a constant gradient the corrected moments are exact at *every* $k$, which is
the cleanest possible confirmation of the bias-correction formula.

$$
\boxed{\Delta_1^{\text{corrected}} = -\alpha = -10^{-3}; \qquad \Delta_1^{\text{raw}} = -3.16\,\alpha}
$$

> **Key takeaway:** Adam's step size is (in the noiseless limit) *scale-invariant*: multiplying the loss by $1000$ leaves the trajectory unchanged, which is why a single $\alpha$ can serve layers whose gradient magnitudes differ by orders of magnitude.

### Problem L1.6: The Effective Step Size and Averaging Window of Momentum

**Problem Statement:** For $\mathbf{m}_{k+1} = \beta\mathbf{m}_k + \mathbf{g}_k$,
$\mathbf{x}_{k+1} = \mathbf{x}_k - \alpha\mathbf{m}_{k+1}$, derive the steady-state step size for a constant
gradient and the variance-reduction factor for i.i.d. noise. Evaluate both at $\beta = 0,\ 0.9,\ 0.99$ and
explain the practical rule "when you raise $\beta$, lower $\alpha$".

*Intuition:* Momentum accumulates a geometric series of gradients, which multiplies the signal by $1/(1-\beta)$ while averaging the noise over a window of comparable length.

**Solution:**

**Step 1 (unroll).** With $\mathbf{m}_0 = \mathbf{0}$,

$$
\mathbf{m}_{k} = \sum_{t=1}^{k}\beta^{\,k-t}\mathbf{g}_t
$$

**Step 2 (constant gradient: the effective step).** If $\mathbf{g}_t \equiv \mathbf{g}$, then
$\mathbf{m}_k = \mathbf{g}\sum_{j=0}^{k-1}\beta^j \to \frac{\mathbf{g}}{1-\beta}$, so the steady-state
displacement per iteration is

$$
\alpha_{\mathrm{eff}} = \frac{\alpha}{1-\beta}
$$

**Step 3 (i.i.d. noise: the averaging window).** If $\mathbf{g}_t = \boldsymbol{\epsilon}_t$ with
i.i.d. mean-zero noise of variance $\sigma_B^2$, then for large $k$

$$
\operatorname{Var}\left[(1-\beta)\mathbf{m}_k\right] = (1-\beta)^2\sigma_B^2\sum_{j\ge0}\beta^{2j} = \frac{(1-\beta)^2}{1-\beta^2}\sigma_B^2 = \frac{1-\beta}{1+\beta}\,\sigma_B^2
$$

so the normalized buffer behaves like an average of

$$
n_{\mathrm{eff}} = \frac{1+\beta}{1-\beta} \ \text{samples}
$$

**Step 4 (the numbers).**

| $\beta$ | $\alpha_{\mathrm{eff}}/\alpha = \frac{1}{1-\beta}$ | $n_{\mathrm{eff}} = \frac{1+\beta}{1-\beta}$ | noise factor $\frac{1-\beta}{1+\beta}$ |
|---|---|---|---|
| $0$ | $1$ | $1$ | $1$ |
| $0.9$ | $10$ | $19$ | $0.0526$ |
| $0.99$ | $100$ | $199$ | $0.00503$ |

**Step 5 (the practical rule).** Raising $\beta$ from $0.9$ to $0.99$ multiplies the effective step by
$10$ while the stability ceiling is still governed by curvature. To keep $\alpha_{\mathrm{eff}}$ fixed one
must divide $\alpha$ by $10$; skipping that is the classic cause of a run that trained fine at
$\beta = 0.9$ and diverged at $\beta = 0.99$. The compensating benefit is the $10\times$ larger averaging
window, which reduces the noise ball at fixed $\alpha_{\mathrm{eff}}$ — momentum is simultaneously an
accelerator (Topic 03) and a variance reducer.

$$
\boxed{\alpha_{\mathrm{eff}} = \frac{\alpha}{1-\beta}, \qquad n_{\mathrm{eff}} = \frac{1+\beta}{1-\beta}, \qquad \text{noise variance} \times \frac{1-\beta}{1+\beta}}
$$

> **Key takeaway:** $\beta$ and $\alpha$ are not independent hyperparameters — only the product $\alpha/(1-\beta)$ enters the deterministic dynamics, so tuning should be done in that coordinate.

## Level 2 — Applications in AI/ML & Physics

### Problem L2.1: Gradient Evaluations to Target Accuracy — GD vs SGD vs SVRG

**Problem Statement:** A strongly convex finite-sum problem has $N = 10^6$, $\kappa = L/\mu = 100$, and
SGD constants with $G^2/\mu^2 = 10$. Count total *per-example gradient evaluations* to reach
$\epsilon = 10^{-3}$ and $\epsilon = 10^{-8}$ for (i) full-batch GD, (ii) SGD, (iii) SVRG. Identify the
crossover.

*Intuition:* GD pays $N$ per step but converges geometrically; SGD pays $1$ per step but only sublinearly; SVRG buys the geometric rate at nearly SGD's per-step cost.

**Solution:**

**Step 1 (the three complexities).**

$$
\text{GD}: \ N\,\kappa\ln\frac{1}{\epsilon}, \qquad \text{SGD}: \ \frac{G^2}{\mu^2\epsilon}, \qquad \text{SVRG}: \ (N+\kappa)\ln\frac{1}{\epsilon}
$$

(GD needs $\kappa\ln(1/\epsilon)$ iterations at $N$ gradients each; SGD needs $O(1/\epsilon)$ iterations at
$1$ gradient each; SVRG needs $O(\log(1/\epsilon))$ epochs each costing $O(N+\kappa)$.)

**Step 2 ($\epsilon = 10^{-3}$, $\ln(1/\epsilon) = 6.91$).**

| Method | Computation | Gradient evaluations |
|---|---|---|
| GD | $10^6\times100\times6.91$ | $6.9\times10^{8}$ |
| SGD | $10/10^{-3}$ | $1.0\times10^{4}$ |
| SVRG | $(10^6+100)\times6.91$ | $6.9\times10^{6}$ |

SGD wins by **five orders of magnitude** over GD at loose accuracy: it reaches $10^{-3}$ before GD has
finished a single pass over the data ($10^6$ evaluations).

**Step 3 ($\epsilon = 10^{-8}$, $\ln(1/\epsilon) = 18.42$).**

| Method | Computation | Gradient evaluations |
|---|---|---|
| GD | $10^6\times100\times18.42$ | $1.8\times10^{9}$ |
| SGD | $10/10^{-8}$ | $1.0\times10^{9}$ |
| SVRG | $(10^6+100)\times18.42$ | $1.8\times10^{7}$ |

Now SGD and GD are comparable, and SVRG beats both by two orders of magnitude.

**Step 4 (the crossover).** SGD beats SVRG when

$$
\frac{G^2}{\mu^2\epsilon} \lt (N+\kappa)\ln\frac{1}{\epsilon} \ \Longrightarrow\ \epsilon \gtrsim \frac{G^2/\mu^2}{N\ln(1/\epsilon)} \approx \frac{10}{10^6\times14} \approx 7\times10^{-7}
$$

So for the loose accuracies typical of deep learning ($\epsilon \sim 10^{-2}$ to $10^{-4}$ in generalization
terms), plain SGD is the right tool; for high-accuracy convex problems, variance reduction wins.

**Step 5 (why deep learning still uses SGD).** Generalization error saturates long before optimization
error reaches $10^{-8}$: driving the *training* loss further buys nothing on held-out data, and SVRG's
snapshot cost ($N$ gradients per epoch) plus its extra memory rarely pay for themselves.

$$
\boxed{\epsilon = 10^{-3}: \ \text{SGD } 10^4 \ \text{vs GD } 6.9\times10^8; \quad \epsilon = 10^{-8}: \ \text{SVRG } 1.8\times10^7 \ \text{wins}}
$$

> **Key takeaway:** There is no universally best stochastic method — the ranking flips with the target accuracy, and machine learning lives permanently in the loose-accuracy regime where plain SGD is optimal.

### Problem L2.2: The Linear Scaling Rule and Critical Batch Size

**Problem Statement:** A model trains stably at $B = 256$ with $\alpha = 0.1$. You move to $B = 8192$ on a
cluster. State the learning rate the linear scaling rule prescribes, check it against the stability limit
$\alpha \lt 2/L$ with $L = 50$, and decide whether the change is worthwhile if the measured gradient noise
scale is $\mathcal{B}_{\text{noise}} = 2000$.

*Intuition:* Only the ratio $\alpha/B$ controls the noise floor, so scaling both together preserves the training dynamics — until curvature or the noise scale intervenes.

**Solution:**

**Step 1 (the rule).** The noise floor is $\propto \alpha\sigma^2/B$, so keeping $\alpha/B$ constant
preserves it:

$$
B: 256 \to 8192 \ (c = 32) \quad\Longrightarrow\quad \alpha: 0.1 \to 32\times0.1 = 3.2
$$

**Step 2 (stability check).** The deterministic stability ceiling is

$$
\alpha \lt \frac{2}{L} = \frac{2}{50} = 0.04
$$

and $3.2 \gg 0.04$: the prescribed rate is **far above** the ceiling and the run would diverge instantly.
(Even the original $\alpha = 0.1$ exceeds $0.04$, which tells us the effective local $L$ at the operating
point is smaller than the global bound — a common situation, and precisely why warmup exists.)

**Step 3 (what actually happens).** The correct statement of the scaling rule is: increase $\alpha$
linearly **as long as** $\alpha$ stays below the curvature ceiling. Once $c\alpha \gt 2/L$, the extra
batch cannot be converted into progress; you have entered the *curvature-limited* regime, and the run
must fall back to $\alpha \approx \alpha_{\max}$ with a floor that is now lower than needed.

**Step 4 (the noise-scale test).** The gradient noise scale
$\mathcal{B}_{\text{noise}} = \frac{\operatorname{tr}\Sigma}{\lVert \nabla f\rVert^2} = 2000$ is the batch
size at which gradient noise and signal are comparable. The empirical law for steps-to-target is

$$
\frac{\text{steps}(B)}{\text{steps}_{\min}} \approx 1 + \frac{\mathcal{B}_{\text{noise}}}{B}
$$

At $B = 256$: factor $1 + 2000/256 = 8.8$. At $B = 8192$: factor $1 + 2000/8192 = 1.24$. So the step count
falls only by $8.8/1.24 \approx 7.1\times$ while the per-step cost rises by $32\times$:

$$
\text{total compute} \times \frac{32}{7.1} \approx 4.5
$$

**Step 5 (verdict).** $B = 8192 \gg \mathcal{B}_{\text{noise}} = 2000$, so the move is **compute-inefficient
by roughly $4.5\times$** — justified only if wall-clock time matters more than total FLOPs (the usual
reason: $32\times$ more parallel hardware). A batch near $\mathcal{B}_{\text{noise}}$, say $B = 2048$
with $\alpha \approx \min\{8\times0.1,\ \alpha_{\max}\}$, is the compute-efficient choice.

$$
\boxed{\text{rule gives } \alpha = 3.2 \gg 2/L = 0.04; \ B = 8192 \gg \mathcal{B}_{\text{noise}} = 2000 \implies \approx 4.5\times \text{ compute waste}}
$$

> **Key takeaway:** Large batches buy *time*, not *compute*; the noise scale tells you where the free lunch ends, and the curvature ceiling tells you when the compensating learning rate becomes unusable.

### Problem L2.3: Designing a Warmup and Decay Schedule

**Problem Statement:** Train for $90$ epochs on $N = 1.28\times10^6$ images at $B = 8192$, targeting a peak
learning rate of $\alpha_{\text{peak}} = 3.2$ from a base of $\alpha_{\text{base}} = 0.1$ at $B = 256$. Use
$5$ epochs of linear warmup then cosine decay to zero. Compute the steps per epoch, the total steps, the
learning rate at step $390$, and at epoch $50$.

*Intuition:* Warmup exists because the curvature at initialization is far larger than at the operating point, so the large-batch learning rate must be approached, not applied.

**Solution:**

**Step 1 (steps per epoch and total).**

$$
S_{\text{epoch}} = \left\lceil\frac{N}{B}\right\rceil = \left\lceil\frac{1.28\times10^6}{8192}\right\rceil = 157 \ (\text{exactly } 156.25), \qquad S_{\text{total}} = 90\times156.25 \approx 1.41\times10^4
$$

Warmup lasts $S_{\text{warm}} = 5\times156.25 \approx 781$ steps.

**Step 2 (warmup schedule).** Linear from $\alpha_{\text{base}}$ to $\alpha_{\text{peak}}$:

$$
\alpha(s) = \alpha_{\text{base}} + \left(\alpha_{\text{peak}}-\alpha_{\text{base}}\right)\frac{s}{S_{\text{warm}}}, \qquad 0 \le s \le S_{\text{warm}}
$$

At $s = 390$ (half of warmup):

$$
\alpha(390) = 0.1 + (3.2-0.1)\frac{390}{781} = 0.1 + 3.1\times0.4994 = 0.1 + 1.548 = 1.648
$$

**Step 3 (cosine decay).** For $s \gt S_{\text{warm}}$, with $\tau = \frac{s - S_{\text{warm}}}{S_{\text{total}}-S_{\text{warm}}}$:

$$
\alpha(s) = \frac{\alpha_{\text{peak}}}{2}\left(1 + \cos(\pi\tau)\right)
$$

**Step 4 (epoch 50).** Step $s = 50\times156.25 = 7813$, so

$$
\tau = \frac{7813 - 781}{14063-781} = \frac{7032}{13282} = 0.5294
$$

$$
\alpha = \frac{3.2}{2}\left(1+\cos(0.5294\pi)\right) = 1.6\left(1 - 0.0922\right) = 1.452
$$

(using $\cos(0.5294\pi) = \cos(95.3^\circ) \approx -0.0922$).

**Step 5 (why this shape).**

- **Warmup** keeps $\alpha \lt 2/L_{\text{local}}$ while the initialization's curvature is large; skipping
  it with $\alpha = 3.2$ diverges within a handful of steps.
- **A long plateau near the peak** maximizes $\sum_k\alpha_k$, i.e. total distance travelled — the first
  Robbins-Monro requirement.
- **Decay to zero** collapses the noise ball ($\delta_\infty\propto\alpha$), which is what converts the
  plateau into a final loss drop. The cosine shape simply spends more steps at high $\alpha$ than a linear
  decay while still reaching zero.

$$
\boxed{S_{\text{epoch}} \approx 156,\ S_{\text{total}} \approx 1.41\times10^4,\ \alpha(390) = 1.648,\ \alpha(\text{epoch }50) = 1.452}
$$

> **Key takeaway:** A modern schedule is the Robbins-Monro conditions plus a stability constraint, expressed in engineering form: ramp up under the curvature ceiling, cruise to accumulate distance, decay to quiet the noise.

### Problem L2.4: SGD versus Adam on an Ill-Conditioned Quadratic

**Problem Statement:** Let $f(\mathbf{x}) = \frac12\mathbf{x}^TA\mathbf{x}$ with
$A = \operatorname{diag}(1, 100)$ and isotropic gradient noise of variance $\sigma^2 = 1$ per coordinate
with $B=1$. For SGD: give the stability limit, the per-digit iteration count at $\alpha = 0.01$, and the
per-coordinate noise floor. Then explain what Adam changes and what it does not.

*Intuition:* SGD must serve two curvatures with one step size; Adam gives each coordinate its own.

**Solution:**

**Step 1 (SGD stability).** $L = \lambda_{\max} = 100$, so

$$
0 \lt \alpha \lt \frac{2}{L} = 0.02
$$

Take $\alpha = 0.01$ (half the ceiling).

**Step 2 (transient rate).** Each coordinate is an independent AR(1) with $\rho_j = 1-\alpha\lambda_j$:

$$
\rho_1 = 1 - 0.01 = 0.99, \qquad \rho_2 = 1 - 1 = 0
$$

The stiff coordinate converges in **one step**; the flat one contracts by $0.99$, needing

$$
k = \frac{\ln 10}{-\ln 0.99} \approx 229 \ \text{iterations per decimal digit}
$$

The whole run is dictated by $\kappa = 100$, exactly as the deterministic theory predicts.

**Step 3 (per-coordinate noise floor).** From the AR(1) stationary variance
$v_j = \frac{\alpha^2\sigma^2}{1-\rho_j^2} = \frac{\alpha^2\sigma^2}{2\alpha\lambda_j - \alpha^2\lambda_j^2}$:

| coordinate | $\lambda_j$ | $\rho_j$ | $v_j$ |
|---|---|---|---|
| $1$ | $1$ | $0.99$ | $\frac{10^{-4}}{1-0.9801} = 5.03\times10^{-3}$ |
| $2$ | $100$ | $0$ | $\frac{10^{-4}}{1} = 1.0\times10^{-4}$ |

The flat coordinate carries $\approx 50\times$ more residual variance: SGD is both *slower* and *noisier*
along the direction of small curvature.

**Step 4 (what Adam changes).** Adam divides coordinate $j$ by $\sqrt{\hat{v}_j}$, an estimate of the RMS
gradient in that coordinate. In the signal-dominated (transient) regime the gradient in coordinate $j$ is
$\lambda_j x_j$, so the update magnitude is

$$
\alpha\frac{\lambda_j\lvert x_j\rvert}{\lambda_j\lvert x_j\rvert} = \alpha \quad \text{for both } j
$$

Every coordinate advances by $\approx\alpha$ per step, so the *effective* condition number is $\approx1$
and the transient no longer scales with $\kappa$: reaching $\lvert x_1\rvert \sim 10^{-3}$ from
$\lvert x_1\rvert\sim1$ takes $O(1/\alpha)$ steps rather than $O(\kappa/\alpha)$.

**Step 5 (what Adam does not change).** Once the noise dominates, $\sqrt{\hat{v}_j}\to\sigma$ for every
$j$ and the preconditioner degenerates to a uniform $1/\sigma$: Adam then behaves like SGD with step
$\alpha/\sigma$ and has its own noise ball proportional to $\alpha$. Adam removes the *conditioning*
penalty, not the *noise floor* — which is why Adam runs still need a decay schedule, and why well-tuned
SGD with momentum can match Adam on well-conditioned problems while losing badly on transformers, whose
per-layer gradient scales differ enormously.

$$
\boxed{\text{SGD: } \alpha \lt 0.02, \ 229 \ \text{iters/digit}, \ v_1 = 5.0\times10^{-3} \gg v_2 = 1.0\times10^{-4}; \ \text{Adam: transient } \kappa\text{-free, same } O(\alpha) \ \text{floor}}
$$

> **Key takeaway:** Adaptive methods are diagonal preconditioners — they attack the condition number, which is a *transient* effect, and leave the stochastic floor, which is a *stationary* effect, entirely intact.

### Problem L2.5: The Loss Floor Imposed by Label Noise

**Problem Statement:** Fit a linear model to $y_i = \mathbf{w}^{*T}\mathbf{a}_i + \nu_i$ where
$\nu_i$ is independent label noise with variance $\sigma_\nu^2 = 0.25$, features are standardized with
$\mathbb{E}\lVert \mathbf{a}\rVert^2 = n = 100$, and the loss is $\frac12(\hat{y}-y)^2$ per example. Compute
the per-example gradient variance at $\mathbf{w}^*$, the SGD noise floor with $B = 64$, $\alpha = 0.01$,
$L = 3$, $\mu = 0.5$, and explain why no schedule removes the *statistical* floor.

*Intuition:* At the true parameters the gradient is not zero for any individual example — it is the residual times the feature vector, and residuals are pure noise.

**Solution:**

**Step 1 (per-example gradient at the optimum).** With $f_i(\mathbf{w}) = \frac12(\mathbf{w}^T\mathbf{a}_i - y_i)^2$,

$$
\nabla f_i(\mathbf{w}^*) = \left(\mathbf{w}^{*T}\mathbf{a}_i - y_i\right)\mathbf{a}_i = -\nu_i\,\mathbf{a}_i
$$

The population gradient is $\mathbb{E}[-\nu\mathbf{a}] = \mathbf{0}$ (noise independent of features), so
$\mathbf{w}^*$ is stationary — but every individual gradient is nonzero.

**Step 2 (gradient variance).**

$$
\sigma^2 = \mathbb{E}\left\lVert \nu\mathbf{a}\right\rVert^2 = \mathbb{E}\left[\nu^2\right]\,\mathbb{E}\left\lVert \mathbf{a}\right\rVert^2 = 0.25\times100 = 25
$$

**Step 3 (batch variance and noise floor).**

$$
\sigma_B^2 = \frac{25}{64} = 0.3906, \qquad \delta_\infty = \frac{L\alpha\sigma_B^2}{2\mu} = \frac{3\times0.01\times0.3906}{2\times0.5} = 0.0117
$$

So constant-step SGD hovers about $1.2\times10^{-2}$ above the minimum of the *empirical* risk.

**Step 4 (two different floors).** There are two distinct, non-interchangeable floors:

| Floor | Value | Removable? |
|---|---|---|
| **Optimization floor** $\delta_\infty$ | $0.0117$ | Yes — decay $\alpha$, raise $B$, use SVRG |
| **Statistical floor** (irreducible loss) | $\frac12\sigma_\nu^2 = 0.125$ | No — no optimizer can predict $\nu_i$ |

Even a perfect optimizer that returns $\mathbf{w}^*$ exactly has expected loss $\frac12\sigma_\nu^2 = 0.125$
on fresh data. Here the optimization floor is $10\times$ *smaller* than the statistical floor.

**Step 5 (the practical conclusion).** Since $0.0117 \ll 0.125$, further optimization effort is wasted:
tightening $\alpha$ improves training loss but cannot improve test loss, which is dominated by
$\sigma_\nu^2$. Driving the *empirical* risk below $0.125$ on the training set means fitting the noise
$\nu_i$ — memorization, visible as a widening train/test gap. This is the precise sense in which the loose
accuracy regime of Problem L2.1 is not a compromise but the correct target.

$$
\boxed{\sigma^2 = \sigma_\nu^2 n = 25, \quad \delta_\infty = 0.0117, \quad \text{irreducible loss} = \tfrac12\sigma_\nu^2 = 0.125}
$$

> **Key takeaway:** Compare your optimization floor with your statistical floor before tuning: once the former is well below the latter, every additional optimizer improvement is invisible on held-out data.

### Problem L2.6: SGD as Langevin Dynamics — Temperature and Escape Times

**Problem Statement:** Model SGD as overdamped Langevin dynamics with temperature
$T = \frac{\alpha\sigma^2}{B}$. For $\sigma^2 = 1$ and a basin separated from a better one by a barrier
$\Delta E = 0.05$, compute $T$ and the Kramers escape time $\tau \sim e^{2\Delta E/T}$ for
$(\alpha, B) = (0.1,\ 32)$ and $(0.1,\ 4)$. Explain the implication for batch size and generalization.

*Intuition:* Only the ratio $\alpha/B$ sets the temperature, and escape times depend on it exponentially — so a modest change in batch size changes trapping behaviour by many orders of magnitude.

**Solution:**

**Step 1 (the diffusion).** Writing $\mathbf{g}_k = \nabla f + \boldsymbol{\epsilon}_k$ with
$\operatorname{Cov}(\boldsymbol{\epsilon}_k) = \frac{\sigma^2}{B}I$, one SGD step is an Euler-Maruyama step
of

$$
d\mathbf{x} = -\nabla f(\mathbf{x})\,dt + \sqrt{T}\,d\mathbf{W}_t, \qquad T = \frac{\alpha\sigma^2}{B}
$$

with stationary density $p(\mathbf{x})\propto\exp\left(-\frac{2f(\mathbf{x})}{T}\right)$.

**Step 2 (case $B = 32$).**

$$
T = \frac{0.1\times1}{32} = 3.125\times10^{-3}, \qquad \frac{2\Delta E}{T} = \frac{0.1}{3.125\times10^{-3}} = 32
$$

$$
\tau \sim e^{32} \approx 7.9\times10^{13} \ \text{steps}
$$

Effectively **trapped**: at $10^4$ steps per second this is $250$ years.

**Step 3 (case $B = 4$).**

$$
T = \frac{0.1}{4} = 0.025, \qquad \frac{2\Delta E}{T} = \frac{0.1}{0.025} = 4, \qquad \tau \sim e^{4} \approx 55 \ \text{steps}
$$

Escapes almost immediately. An eightfold change in $B$ changed the escape time by **twelve orders of
magnitude** — the exponential dependence is the whole story.

**Step 4 (flat minima).** For a quadratic basin with Hessian $H$, the Gibbs mass of the basin is

$$
\int_{\text{basin}}e^{-2f/T} \propto \left(\det H\right)^{-1/2}\,T^{n/2}
$$

so a *flat* basin (small $\det H$) collects exponentially more probability mass than a sharp one of the
same depth. High temperature (small $B$, large $\alpha$) therefore biases SGD toward flat minima, which
empirically generalize better. This is a quantitative version of "small-batch SGD generalizes better".

**Step 5 (the practical dial).**

| Regime | $T = \alpha\sigma^2/B$ | Behaviour |
|---|---|---|
| Early training | high | explores, escapes saddles and sharp basins |
| Late training | low (decayed $\alpha$) | settles into the basin it has found |

Learning-rate decay is literally **simulated annealing**: lower the temperature slowly enough for the
system to equilibrate in a good basin, then freeze. Large-batch training lowers $T$ inadvertently, which is
one accepted explanation of the large-batch generalization gap and a motivation for adding explicit noise
or scaling $\alpha$ up with $B$.

$$
\boxed{B=32: \ T = 3.1\times10^{-3},\ \tau \sim e^{32} \approx 8\times10^{13}; \quad B=4: \ T = 0.025,\ \tau\sim e^{4}\approx55}
$$

> **Key takeaway:** SGD is a thermostatted dynamical system whose temperature you control through $\alpha/B$; every schedule decision is simultaneously an annealing decision with exponential consequences for which basin you land in.

## Level 3 — Challenge

### Problem L3.1: Polyak Averaging Beats the Last Iterate — Exactly

**Problem Statement:** For $f(x) = \frac{\mu}{2}x^2$ with constant step $\alpha$ and i.i.d. gradient noise
of variance $\sigma^2$, compute (a) the stationary variance of the last iterate, and (b) the exact
long-run variance of the average $\bar{x}_K = \frac1K\sum_{k \lt K}x_k$. Show that the latter is
$\frac{\sigma^2}{\mu^2K}$, **independent of $\alpha$**.

*Intuition:* The iterate is an AR(1) process; averaging an AR(1) cancels precisely the $\alpha$-dependence, leaving the information-theoretic optimum.

**Solution:**

**Step 1 (the process).** As in Problem L0.2,

$$
x_{k+1} = \rho\,x_k - \alpha\epsilon_k, \qquad \rho = 1-\alpha\mu, \qquad \operatorname{Var}(\alpha\epsilon_k) = \alpha^2\sigma^2
$$

a stationary AR(1) for $0 \lt \alpha \lt 2/\mu$.

**(a) Last-iterate variance.**

$$
v = \operatorname{Var}(x_\infty) = \frac{\alpha^2\sigma^2}{1-\rho^2} = \frac{\alpha^2\sigma^2}{(1-\rho)(1+\rho)} = \frac{\alpha^2\sigma^2}{\alpha\mu\,(2-\alpha\mu)} \approx \frac{\alpha\sigma^2}{2\mu}
$$

It is **proportional to $\alpha$** and does not decrease with $K$: the last iterate never improves once
stationarity is reached.

**Step 2 (autocovariance).** For a stationary AR(1), $\gamma(h) = \operatorname{Cov}(x_k, x_{k+h}) = v\rho^{\lvert h\rvert}$.

**Step 3 (long-run variance of the average).** For a stationary sequence,

$$
\operatorname{Var}\left(\bar{x}_K\right) = \frac{1}{K}\sum_{h=-(K-1)}^{K-1}\left(1-\frac{\lvert h\rvert}{K}\right)\gamma(h) \ \xrightarrow[K\to\infty]{} \ \frac{S}{K}, \qquad S = \sum_{h=-\infty}^{\infty}\gamma(h)
$$

Summing the two-sided geometric series,

$$
S = v\left(1 + 2\sum_{h\ge1}\rho^h\right) = v\,\frac{1+\rho}{1-\rho}
$$

**Step 4 (the cancellation).** Substituting $v$ from part (a), $1-\rho = \alpha\mu$ and $1+\rho = 2-\alpha\mu$:

$$
S = \frac{\alpha^2\sigma^2}{\alpha\mu(2-\alpha\mu)}\cdot\frac{2-\alpha\mu}{\alpha\mu} = \frac{\alpha^2\sigma^2}{\alpha^2\mu^2} = \frac{\sigma^2}{\mu^2}
$$

The factors $(2-\alpha\mu)$ cancel and both powers of $\alpha$ cancel, leaving

$$
\operatorname{Var}\left(\bar{x}_K\right) \approx \frac{\sigma^2}{\mu^2 K}
$$

**exactly independent of the step size**.

**Step 5 (why this is optimal).** $\frac{\sigma^2}{\mu^2}$ is $H^{-1}\Sigma H^{-1}$ with $H = f''= \mu$ and
$\Sigma = \sigma^2$ — the Cramér-Rao / maximum-likelihood asymptotic covariance. So the averaged SGD iterate
is **asymptotically efficient**: it attains the statistical limit for estimating $x^*$ from $K$ noisy
gradient observations, without knowing $\mu$, without tuning $\alpha$, and at $O(1)$ extra cost per step.
The last iterate, by contrast, is off by the factor $\frac{\alpha\mu K}{2}$ — arbitrarily bad for large
$K$.

$$
\boxed{\operatorname{Var}(x_\infty) \approx \frac{\alpha\sigma^2}{2\mu} \ \text{(constant in } K); \qquad \operatorname{Var}(\bar{x}_K) \to \frac{\sigma^2}{\mu^2K} \ \text{(free of } \alpha)}
$$

> **Key takeaway:** Averaging converts a step-size-dependent noise ball into a step-size-free $1/K$ decay — which is why weight averaging (EMA, SWA) so reliably improves models at zero training cost.

### Problem L3.2: The $O(1/\sqrt{K})$ Rate for Convex SGD via the Averaged Iterate

**Problem Statement:** Let $f$ be convex with $\mathbb{E}\lVert \mathbf{g}_k\rVert^2 \le G^2$ and
$\lVert \mathbf{x}_0-\mathbf{x}^*\rVert \le D$. Prove

$$
\mathbb{E}\left[f(\bar{\mathbf{x}}_K)\right]-f^* \le \frac{D^2 + G^2\sum_{k}\alpha_k^2}{2\sum_k\alpha_k}, \qquad \bar{\mathbf{x}}_K = \frac{\sum_k\alpha_k\mathbf{x}_k}{\sum_k\alpha_k}
$$

and optimize over constant $\alpha$ to obtain the $O(GD/\sqrt{K})$ rate.

*Intuition:* Convexity converts each step's distance decrease into a bound on the suboptimality gap, and Jensen moves the bound from an average of values to the value at the average.

**Solution:**

**Step 1 (one-step distance recursion).**

$$
\lVert \mathbf{x}_{k+1}-\mathbf{x}^*\rVert^2 = \lVert \mathbf{x}_k-\mathbf{x}^*\rVert^2 - 2\alpha_k\mathbf{g}_k^T\left(\mathbf{x}_k-\mathbf{x}^*\right) + \alpha_k^2\lVert \mathbf{g}_k\rVert^2
$$

**Step 2 (condition and apply convexity).** Taking $\mathbb{E}[\cdot\mid\mathcal{F}_k]$ and using
unbiasedness,

$$
\mathbb{E}\left[\mathbf{g}_k^T(\mathbf{x}_k-\mathbf{x}^*)\mid\mathcal{F}_k\right] = \nabla f(\mathbf{x}_k)^T\left(\mathbf{x}_k-\mathbf{x}^*\right) \ge f(\mathbf{x}_k)-f^*
$$

the last step being the first-order characterization of convexity. Hence, with total expectations,

$$
\mathbb{E}\lVert \mathbf{x}_{k+1}-\mathbf{x}^*\rVert^2 \le \mathbb{E}\lVert \mathbf{x}_k-\mathbf{x}^*\rVert^2 - 2\alpha_k\left(\mathbb{E}\left[f(\mathbf{x}_k)\right]-f^*\right) + \alpha_k^2G^2
$$

**Step 3 (telescope).** Rearranging and summing $k = 0,\dots,K-1$, the distance terms telescope:

$$
2\sum_{k=0}^{K-1}\alpha_k\left(\mathbb{E}[f(\mathbf{x}_k)]-f^*\right) \le \lVert \mathbf{x}_0-\mathbf{x}^*\rVert^2 - \mathbb{E}\lVert \mathbf{x}_K-\mathbf{x}^*\rVert^2 + G^2\sum_{k=0}^{K-1}\alpha_k^2 \le D^2 + G^2\sum_k\alpha_k^2
$$

**Step 4 (Jensen).** Since $f$ is convex and $\bar{\mathbf{x}}_K$ is the $\alpha$-weighted average,

$$
f(\bar{\mathbf{x}}_K) \le \frac{\sum_k\alpha_kf(\mathbf{x}_k)}{\sum_k\alpha_k}
$$

Combining with Step 3 gives the claimed bound. $\blacksquare$

**Step 5 (optimize a constant step).** With $\alpha_k \equiv \alpha$, the bound reads

$$
\mathbb{E}\left[f(\bar{\mathbf{x}}_K)\right]-f^* \le \frac{D^2}{2\alpha K} + \frac{\alpha G^2}{2}
$$

Minimizing over $\alpha$ (the two terms balance) gives

$$
\alpha^\star = \frac{D}{G\sqrt{K}}, \qquad \mathbb{E}\left[f(\bar{\mathbf{x}}_K)\right]-f^* \le \frac{GD}{\sqrt{K}}
$$

**Step 6 (remarks).**

- The optimal $\alpha$ depends on the *horizon* $K$ — the reason "train longer with the same schedule"
  is not a valid strategy, and the reason anytime schedules use $\alpha_k \propto 1/\sqrt{k}$ (which loses
  only a $\log K$ factor, per Problem L1.3).
- The rate $O(1/\sqrt{K})$ is minimax optimal for stochastic convex optimization, and unlike the
  deterministic case it cannot be accelerated to $O(1/K^2)$: acceleration reduces the *bias* term but leaves
  the variance term $\alpha G^2/2$ untouched.
- The bound is on the *averaged* iterate; the last iterate is provably worse by a $\log K$ factor in the
  worst case (Harvey et al., 2019), reinforcing Problem L3.1.

$$
\boxed{\mathbb{E}\left[f(\bar{\mathbf{x}}_K)\right]-f^* \le \frac{D^2+G^2\sum\alpha_k^2}{2\sum\alpha_k} \ \xrightarrow{\alpha = D/(G\sqrt{K})}\ \frac{GD}{\sqrt{K}}}
$$

> **Key takeaway:** The whole convex stochastic analysis is one telescoping inequality plus Jensen; the two terms of the final bound are literally "distance to travel" and "noise energy spent", and tuning $\alpha$ is balancing them.

### Problem L3.3: Stochastic Mirror Descent on the Simplex Is Exponentiated Gradient

**Problem Statement:** Derive the stochastic mirror descent update for minimizing a convex $f$ over the
probability simplex $\Delta_n$ using the negative-entropy mirror map
$\psi(\mathbf{p}) = \sum_i p_i\ln p_i$. Show the update is multiplicative, and compare its dimension
dependence with projected SGD.

*Intuition:* Choosing the geometry to match the constraint set turns a projection problem into a closed-form reweighting — and replaces a $\sqrt{n}$ by a $\sqrt{\log n}$.

**Solution:**

**Step 1 (the mirror descent step).** Mirror descent replaces the Euclidean proximal step by a Bregman one:

$$
\mathbf{p}_{k+1} = \arg\min_{\mathbf{p}\in\Delta_n}\left\{\alpha\,\mathbf{g}_k^T\mathbf{p} + D_\psi\left(\mathbf{p},\mathbf{p}_k\right)\right\}
$$

with the Bregman divergence of the negative entropy being the KL divergence:

$$
D_\psi\left(\mathbf{p},\mathbf{q}\right) = \sum_i p_i\ln\frac{p_i}{q_i} = \mathrm{KL}\left(\mathbf{p}\,\Vert\,\mathbf{q}\right)
$$

**Step 2 (solve with a multiplier).** With $\mathcal{L} = \alpha\mathbf{g}^T\mathbf{p} + \sum_ip_i\ln\frac{p_i}{p_{k,i}} + \lambda\left(\sum_ip_i-1\right)$,

$$
\frac{\partial\mathcal{L}}{\partial p_i} = \alpha g_{k,i} + \ln\frac{p_i}{p_{k,i}} + 1 + \lambda = 0 \ \Longrightarrow\ p_i = p_{k,i}\,e^{-\alpha g_{k,i}}\,e^{-1-\lambda}
$$

**Step 3 (normalize).** The multiplier is fixed by $\sum_ip_i = 1$, giving the **exponentiated gradient**
(multiplicative weights) update

$$
p_{k+1,i} = \frac{p_{k,i}\,e^{-\alpha g_{k,i}}}{\sum_j p_{k,j}\,e^{-\alpha g_{k,j}}}
$$

Nonnegativity holds automatically ($p_i \gt 0$ for all $k$), so the simplex constraint requires **no
projection at all** — the geometry enforces it.

**Step 4 (the rate and its dimension dependence).** The standard mirror descent bound is

$$
\mathbb{E}\left[f(\bar{\mathbf{p}}_K)\right]-f^* \le \frac{D_\psi\left(\mathbf{p}^*,\mathbf{p}_0\right)}{\alpha K} + \frac{\alpha G_\infty^2}{2}
$$

where $G_\infty$ bounds $\lVert \mathbf{g}\rVert_\infty$ (the norm dual to the $\ell_1$ geometry in which
$\psi$ is strongly convex, by Pinsker's inequality). Starting from the uniform $\mathbf{p}_0$,

$$
D_\psi\left(\mathbf{p}^*,\mathbf{p}_0\right) = \mathrm{KL}\left(\mathbf{p}^*\,\Vert\,\mathbf{u}\right) \le \ln n
$$

Optimizing $\alpha = \sqrt{2\ln n/(G_\infty^2K)}$,

$$
\mathbb{E}\left[f(\bar{\mathbf{p}}_K)\right]-f^* \le G_\infty\sqrt{\frac{2\ln n}{K}}
$$

**Step 5 (comparison with projected SGD).** Projected SGD uses the Euclidean geometry, giving
$\frac{G_2D_2}{\sqrt K}$ with $D_2 = \max\lVert \mathbf{p}-\mathbf{p}_0\rVert_2 \le \sqrt2$ and
$G_2 \le \sqrt{n}\,G_\infty$ in the worst case, hence

| Method | Rate | Per-step cost on $\Delta_n$ |
|---|---|---|
| Projected SGD (Euclidean) | $O\!\left(G_\infty\sqrt{n/K}\right)$ | simplex projection, $O(n\log n)$ |
| Mirror descent (entropic) | $O\!\left(G_\infty\sqrt{\ln n/K}\right)$ | multiplicative update, $O(n)$ |

an improvement from $\sqrt{n}$ to $\sqrt{\ln n}$ — for $n = 10^6$, a factor $\sqrt{10^6/13.8} \approx 269$.

$$
\boxed{p_{k+1,i} \propto p_{k,i}e^{-\alpha g_{k,i}}, \qquad \text{rate } G_\infty\sqrt{\frac{2\ln n}{K}} \ \text{versus} \ O\!\left(G_\infty\sqrt{n/K}\right)}
$$

> **Key takeaway:** The Euclidean geometry is a choice, not a law; matching the mirror map to the constraint set can remove a factor of $\sqrt{n}$ from the rate and eliminate the projection step entirely — this single idea underlies Hedge, AdaBoost, and much of online learning.

### Problem L3.4: Noise Escapes Strict Saddles — An Escape-Time Calculation

**Problem Statement:** Near a strict saddle, let the error along the most negative curvature direction obey
$e_{k+1} = (1+\alpha\gamma)e_k + \alpha\epsilon_k$ with $\gamma \gt 0$ the magnitude of the negative
eigenvalue, $e_0 = 0$, and $\mathbb{E}[\epsilon_k^2] = \sigma^2$. Compute $\mathbb{E}[e_k^2]$ and the number
of steps to reach a distance $r$, and comment on the noiseless case.

*Intuition:* Negative curvature amplifies whatever component exists; noise guarantees a component exists, and the escape time is only logarithmic in how small the noise is.

**Solution:**

**Step 1 (variance recursion).** With $\theta = 1+\alpha\gamma \gt 1$ and $\epsilon_k$ independent of $e_k$,

$$
\mathbb{E}\left[e_{k+1}^2\right] = \theta^2\,\mathbb{E}\left[e_k^2\right] + \alpha^2\sigma^2
$$

**Step 2 (solve with $e_0 = 0$).** This is a geometric recursion:

$$
\mathbb{E}\left[e_k^2\right] = \alpha^2\sigma^2\sum_{j=0}^{k-1}\theta^{2j} = \alpha^2\sigma^2\,\frac{\theta^{2k}-1}{\theta^2-1}
$$

Since $\theta^2 - 1 = 2\alpha\gamma + \alpha^2\gamma^2 \approx 2\alpha\gamma$ for small $\alpha$,

$$
\mathbb{E}\left[e_k^2\right] \approx \frac{\alpha\sigma^2}{2\gamma}\left(\theta^{2k}-1\right)
$$

**Step 3 (escape time).** Escape means $\mathbb{E}[e_k^2] \ge r^2$, i.e.

$$
\theta^{2k} \ge 1 + \frac{2\gamma r^2}{\alpha\sigma^2} \ \Longrightarrow\ k \ge \frac{\ln\!\left(1 + \frac{2\gamma r^2}{\alpha\sigma^2}\right)}{2\ln(1+\alpha\gamma)} \approx \frac{1}{2\alpha\gamma}\ln\!\left(\frac{2\gamma r^2}{\alpha\sigma^2}\right)
$$

using $\ln(1+\alpha\gamma)\approx\alpha\gamma$.

**Step 4 (a numerical instance).** Take $\alpha = 0.01$, $\gamma = 0.1$, $\sigma^2 = 1$, $r = 1$:

$$
k \approx \frac{1}{2(0.01)(0.1)}\ln\!\left(\frac{2(0.1)(1)}{(0.01)(1)}\right) = 500\ln 20 \approx 500\times3.0 = 1498 \ \text{steps}
$$

Shrinking the noise a *millionfold* to $\sigma^2 = 10^{-6}$ gives $500\ln(2\times10^7) \approx 500\times16.8 = 8400$
steps — only $5.6\times$ longer. **Escape time is logarithmic in $1/\sigma^2$**, so even minuscule noise
escapes quickly.

**Step 5 (the noiseless case).** With $\sigma = 0$ and $e_0 = 0$ exactly, the recursion gives $e_k \equiv 0$
forever: deterministic gradient descent initialized exactly on the stable manifold of the saddle never
leaves. That set has measure zero, so generic initialization escapes — but *slowly*, and gradient descent
can be shown to require exponential time to escape certain saddle configurations. Noise converts
"escapes eventually, generically" into "escapes in $O\!\left(\frac{1}{\alpha\gamma}\log\frac{1}{\alpha\sigma^2}\right)$ steps,
always".

**Step 6 (what this explains).** The observed training plateau — flat loss, tiny gradient norm, then a
sudden resumption — is exactly this dynamic: $\mathbb{E}[e_k^2]$ grows geometrically but from a tiny base,
so nothing visible happens for $\Theta(1/(\alpha\gamma))$ steps and then everything happens at once. It is
also why deliberately *adding* noise (perturbed gradient descent, Jin et al. 2017) yields polynomial-time
guarantees for reaching second-order stationary points.

$$
\boxed{\mathbb{E}\left[e_k^2\right] = \alpha^2\sigma^2\frac{(1+\alpha\gamma)^{2k}-1}{(1+\alpha\gamma)^2-1}, \qquad k_{\text{escape}} \approx \frac{1}{2\alpha\gamma}\ln\frac{2\gamma r^2}{\alpha\sigma^2}}
$$

> **Key takeaway:** Gradient noise is not merely tolerable near saddles — it is the mechanism that makes escape *guaranteed and fast*, which is why nonconvex training works at all and why the noise ball is a feature in the transient phase even though it is a nuisance at the end.